## Part 1: COmbine all chunks to a single file and preprocess some of the labels.

In [16]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from shapely.errors import WKTReadingError
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import os

input_dir = "/mnt/disk/data/trajfm_veraset_splits/veraset/Visits/LosAngeles/"

attr_POIs = 0
exact_matches = 0
total_visits = 0
dataset = []
for file in os.listdir(input_dir):
    print(f"Loading {file}")
    if file.endswith("_thres100.parquet"):
        df = pd.read_parquet(os.path.join(input_dir, file))
        # stats
        attr_POIs += df['safegraph.place_id'].notnull().sum()
        exact_matches += df['safegraph.dist_to_poi'].eq(0).sum()
        total_visits += len(df)
        # drop the whole rows when null values in 'safegraph.place_id'
        df = df.dropna(subset=['safegraph.place_id'])
        dataset.append(df)
df = pd.concat(dataset, ignore_index=True)

/tmp/ipykernel_433914/3124480104.py:6: FutureWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import WKTReadingError


Loading part-00009_thres100.parquet
Loading part-00000_thres100.parquet
Loading part-00012_thres100.parquet
Loading subset_veraset_preprocessed.parquet
Loading subset_veraset_processed.parquet
Loading part-00001_thres100.parquet
Loading popular_times.parquet
Loading data_splits
Loading whole_veraset_processed_only_attributed.parquet
Loading part-00005_thres100.parquet
Loading subset_pois_veraset_processed.parquet
Loading part-00004_thres100.parquet
Loading part-00007_thres100.parquet
Loading part-00002_thres100.parquet
Loading part-00003_thres100.parquet
Loading part-00006_thres100.parquet
Loading part-00010_thres100.parquet
Loading whole_veraset_processed.parquet
Loading part-00008_thres100.parquet


In [17]:
print(f"Loaded Veraset dataset with {len(df)} rows.")

Loaded Veraset dataset with 1761806 rows.


In [18]:
print(f"Timespan of Veraset dataset: {df['properties.started_at'].min()} - {df['properties.finished_at'].max()}")

Timespan of Veraset dataset: 2019-01-01T00:00:08Z - 2019-12-31T23:59:27Z


In [19]:
print(f"Total visits: {total_visits}")
print(f"Total POIs with attributes: {attr_POIs}")
if total_visits > 0:
    print(f"Percentage of visits with POI attributes: {attr_POIs / total_visits * 100:.2f}%")
if total_visits > 0:
    print(f"Percentage of exact matches: {exact_matches / total_visits * 100:.2f}%")

Total visits: 7082831
Total POIs with attributes: 1761806
Percentage of visits with POI attributes: 24.87%
Percentage of exact matches: 5.66%


In [20]:
df.columns

Index(['type', 'properties.user_id', 'properties.started_at',
       'properties.finished_at', 'geometry.type', 'geometry.coordinates',
       'geometry.latitude', 'geometry.longitude', 'geometry', 'index_right',
       'safegraph.placekey', 'safegraph.place_id', 'safegraph.parent_placekey',
       'safegraph.parent_place_id', 'safegraph.brand_ids',
       'safegraph.location_name', 'safegraph.brands', 'safegraph.top_category',
       'safegraph.sub_category', 'safegraph.naics_code', 'safegraph.latitude',
       'safegraph.longitude', 'safegraph.street_address', 'safegraph.city',
       'safegraph.region', 'safegraph.postal_code', 'safegraph.open_hours',
       'safegraph.category_tags', 'safegraph.opened_on', 'safegraph.closed_on',
       'safegraph.tracking_opened_since', 'safegraph.tracking_closed_since',
       'safegraph.polygon_wkt', 'safegraph.polygon_class',
       'safegraph.building_height', 'safegraph.enclosed',
       'safegraph.phone_number', 'safegraph.is_synthetic',
    

In [21]:
print("Veraset has {} unique POIs.".format(df['safegraph.place_id'].nunique()))

Veraset has 39595 unique POIs.


In [22]:
print("Veraset has {} unique users.".format(df['properties.user_id'].nunique()))

Veraset has 74519 unique users.


In [9]:
df['properties.user_id'].min(), df['properties.user_id'].max()

('0000baf64ae05cd47c9ab400a8a6ea7ddde91bd90ac162ca94755d2b85a033fb',
 'ffffb8b4bb1b38a369aa83440394ab5228fbea9e575168a885774fe0285c4291')

In [23]:
# group all visits by user_id
user_visits = df.groupby('properties.user_id').size().reset_index(name='visits')

# get stats on total visits per user
user_visits['visits'].describe()

count    74519.000000
mean        23.642373
std        118.740960
min          1.000000
25%          1.000000
50%          3.000000
75%          9.000000
max       5777.000000
Name: visits, dtype: float64

In [25]:
# drop users with less than 5 visits
user_visits = user_visits[user_visits['visits'] > 5]

# keep the visits of the selected users in df
df = df[df['properties.user_id'].isin(user_visits['properties.user_id'])]

In [26]:
# cross check the stats again
user_visits = df.groupby('properties.user_id').size().reset_index(name='visits')
user_visits['visits'].describe()

count    26082.00000
mean        63.86830
std        194.40173
min          6.00000
25%          9.00000
50%         16.00000
75%         40.00000
max       5777.00000
Name: visits, dtype: float64

In [27]:
print("Veraset has {} unique POIs after removing small sequencies.".format(df['safegraph.place_id'].nunique()))

Veraset has 39491 unique POIs after removing small sequencies.


In [13]:
# map user_id to a number
df['user_id'] = df['properties.user_id'].astype('category').cat.codes

/tmp/ipykernel_433914/2369166466.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['user_id'] = df['properties.user_id'].astype('category').cat.codes


In [14]:
df['user_id'].min(), df['user_id'].max()

(np.int16(0), np.int16(26081))

In [15]:
df['place_id'] = df['safegraph.place_id'].astype('category').cat.codes

df['place_id'].min(), df['place_id'].max()

/tmp/ipykernel_433914/1525987330.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['place_id'] = df['safegraph.place_id'].astype('category').cat.codes


(np.int32(0), np.int32(39490))

In [14]:
df['category'] = df['safegraph.top_category'].astype('category').cat.codes + 1  # some POIs do not have category

print(df['category'].min(), df['category'].max())

0 149


/tmp/ipykernel_413781/4117459922.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['category'] = df['safegraph.top_category'].astype('category').cat.codes + 1  # some POIs do not have category


In [15]:
# count number of POIs with category 0
print("Number of POIs with category 0: {}".format((df['category'] == 0).sum()))

Number of POIs with category 0: 1645


In [16]:
# pick 1 POI with category 0 and print the other attributes
print(df[df['category'] == 0].iloc[0])

user_id                                                                           20
lat                                                                        29.737467
long                                                                      -95.467201
arrival_time                                                     2020-03-05 02:57:46
leave_time                                                       2020-03-05 04:23:04
traj_id                                                                          227
stay_time                                                                     5118.0
geometry                           b'\x01\x01\x00\x00\x00%\xea\x05\x9f\xe6\xddW\x...
index_right                                                                      NaN
safegraph.placekey                                               zzw-222@8fc-fbg-s5z
safegraph.place_id                               sg:52de9befa0f44281be22f9a581ab00dc
safegraph.parent_placekey                                        

In [17]:
df['category'].nunique()

150

In [18]:
df['naics_2digit'] = df['safegraph.naics_code'].apply(
    lambda x: str(int(x))[:2] if pd.notnull(x) else None
)

/tmp/ipykernel_413781/1166038781.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['naics_2digit'] = df['safegraph.naics_code'].apply(


In [19]:
df['naics_2digit'].unique()

array(['23', '53', '49', '71', '32', '72', '81', '33', '61', '44', '45',
       '54', '62', '52', '56', '48', None, '51', '92', '31', '42', '55',
       '11', '22'], dtype=object)

In [20]:
df['top_naics_category'] = df['naics_2digit'].astype('category').cat.codes

# replace -1 with max + 1
df['top_naics_category'] = df['top_naics_category'] + 1

/tmp/ipykernel_413781/2282638787.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['top_naics_category'] = df['naics_2digit'].astype('category').cat.codes
/tmp/ipykernel_413781/2282638787.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['top_naics_category'] = df['top_naics_category'] + 1


In [21]:
df['top_naics_category'].value_counts()

top_naics_category
22    38028
8     32967
21    28743
19    21843
20    12084
14    12037
18    11650
13     9356
9      8316
10     5852
15     4122
12     2430
23     2410
7      2194
3      2186
11     1821
0      1645
4      1101
5      1049
17      984
6       771
16      136
2         6
1         3
Name: count, dtype: int64

In [22]:
# create place_lat and place_lon columns. Assign safegraph.latitude and safegraph.longitude to them if not nan else assign geometry.latitude and geometry.longitude

df['place_lat'] = df['safegraph.latitude'].where(
    df['safegraph.latitude'].notnull(), df['lat']
)

df['place_lon'] = df['safegraph.longitude'].where(
    df['safegraph.longitude'].notnull(), df['long']
)

/tmp/ipykernel_413781/1251037445.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['place_lat'] = df['safegraph.latitude'].where(
/tmp/ipykernel_413781/1251037445.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['place_lon'] = df['safegraph.longitude'].where(


In [23]:
df['place_lat'].isnull().sum(), df['place_lon'].isnull().sum()

(np.int64(0), np.int64(0))

In [25]:
df.columns

Index(['user_id', 'lat', 'long', 'arrival_time', 'leave_time', 'traj_id',
       'stay_time', 'geometry', 'index_right', 'safegraph.placekey',
       'safegraph.place_id', 'safegraph.parent_placekey',
       'safegraph.parent_place_id', 'safegraph.brand_ids',
       'safegraph.location_name', 'safegraph.brands', 'safegraph.top_category',
       'safegraph.sub_category', 'safegraph.naics_code', 'safegraph.latitude',
       'safegraph.longitude', 'safegraph.street_address', 'safegraph.city',
       'safegraph.region', 'safegraph.postal_code', 'safegraph.open_hours',
       'safegraph.category_tags', 'safegraph.opened_on', 'safegraph.closed_on',
       'safegraph.tracking_opened_since', 'safegraph.tracking_closed_since',
       'safegraph.polygon_wkt', 'safegraph.polygon_class',
       'safegraph.building_height', 'safegraph.enclosed',
       'safegraph.phone_number', 'safegraph.is_synthetic',
       'safegraph.includes_parking_lot', 'safegraph.iso_country_code',
       'safegraph.dist_to

In [ ]:
# drop 'type' column
df = df.drop(columns=['type', 
                      'index_right', 
                      'geometry', 
                      'geometry.type', 
                      ])

In [27]:
df = df.rename(columns={
    'arrival_time': 'properties.started_at',
    'leave_time': 'properties.finished_at',
})

In [28]:
# rename columns

df = df.rename(columns={
    'properties.user_id': 'user_str',
    'properties.started_at': 'arrival_time',
    'properties.finished_at': 'departure_time',
    'geometry.coordinates': 'gps_coordinates',
})

In [29]:
df.columns

Index(['user_id', 'lat', 'long', 'arrival_time', 'departure_time', 'traj_id',
       'stay_time', 'geometry', 'index_right', 'safegraph.placekey',
       'safegraph.place_id', 'safegraph.parent_placekey',
       'safegraph.parent_place_id', 'safegraph.brand_ids',
       'safegraph.location_name', 'safegraph.brands', 'safegraph.top_category',
       'safegraph.sub_category', 'safegraph.naics_code', 'safegraph.latitude',
       'safegraph.longitude', 'safegraph.street_address', 'safegraph.city',
       'safegraph.region', 'safegraph.postal_code', 'safegraph.open_hours',
       'safegraph.category_tags', 'safegraph.opened_on', 'safegraph.closed_on',
       'safegraph.tracking_opened_since', 'safegraph.tracking_closed_since',
       'safegraph.polygon_wkt', 'safegraph.polygon_class',
       'safegraph.building_height', 'safegraph.enclosed',
       'safegraph.phone_number', 'safegraph.is_synthetic',
       'safegraph.includes_parking_lot', 'safegraph.iso_country_code',
       'safegraph.dis

In [31]:
# save the dataframe to a parquet file
df.to_parquet("/mnt/disk/data/trajfm_veraset_splits/veraset/Visits/Houston/whole_veraset_processed.parquet", index=False)

## Some dataset stats

In [35]:
import pandas as pd

df = pd.read_parquet("/home/Shared/datasets/humo_data/veraset/Visits/LosAngeles/whole_veraset_processed_only_attributed.parquet")

In [2]:
# count the number of unique POIs

df['place_id'].nunique()

39491

In [3]:
# get all place_ids in a list
n_pois = df['place_id'].unique().tolist()

In [4]:
# keep 1000 users
rng = pd.Series(n_pois).sample(frac=1.0, random_state=42)

sample_POIs = rng[:1000].tolist()

In [5]:
# keep only sequences that contain the sampled POIs
df = df[df['place_id'].isin(sample_POIs)]

df['place_id'].nunique()

1000

In [6]:
df.shape

(51815, 42)

In [7]:
df.head()

,user_str,arrival_time,departure_time,gps_coordinates,geometry.latitude,geometry.longitude,safegraph.placekey,safegraph.place_id,safegraph.parent_placekey,safegraph.parent_place_id,...,safegraph.phone_number,safegraph.is_synthetic,safegraph.includes_parking_lot,safegraph.iso_country_code,safegraph.dist_to_poi,user_id,place_id,category,place_lat,place_lon
39,09ae9d8289564ee1335118ac98b6a632b97fa7f47ba681...,2019-01-01T03:36:52Z,2019-01-01T03:41:55Z,"[-118.283581, 33.988798]",33.988798,-118.283581,zzw-227@5z6-3q8-btv,sg:667bf12243f64037afe90b70aa1fabdc,None,None,...,1.323697e+10,False,False,US,37.669860,909,16310,44,33.988899,-118.283191
67,11898e1ea6023ec5a583fd78225eb5479d34d43dfe7561...,2019-01-01T12:02:21Z,2019-01-01T12:07:38Z,"[-118.308937, 34.10155]",34.101550,-118.308937,zzw-222@5z5-3r6-hwk,sg:660731ff05d94ca3915ed8a4bc3adc6c,None,None,...,1.323860e+10,False,None,US,17.977262,1705,16242,37,34.101461,-118.308774
98,196f6edac48c38316cb4135d83071f6dbdd46be433aef1...,2019-01-01T18:56:15Z,2019-01-01T19:03:50Z,"[-118.30177333333336, 33.96008633333333]",33.960086,-118.301773,zzw-224@5z6-3qb-kfz,sg:4753fa700ed24cdc814d082914b7f134,None,None,...,1.323752e+10,False,False,US,27.284515,2580,11445,116,33.960245,-118.301999
145,216f68775bef9b4ed5369c34e028fba933805fff141271...,2019-01-01T04:01:10Z,2019-01-01T04:08:14Z,"[-118.28075, 34.010455]",34.010455,-118.280750,zzw-222@5z5-3qw-fzz,sg:791d5b088ae547ecab3902c2cacd838e,None,None,...,1.323336e+10,False,False,US,51.166036,3383,19104,108,34.010464,-118.280195
275,4624f83b9c4c48a914f860b8c6107e3aaa6bd9479d398b...,2019-01-01T17:44:22Z,2019-01-01T18:09:02Z,"[-118.2606005263158, 34.03932978947368]",34.039330,-118.260601,zzy-223@5z5-3qy-vpv,sg:04cfe2d602e94adda7f72d3b0deb6d49,zzw-222@5z5-3qy-vj9,sg:1c4445a754a84b57ba459ac233070e81,...,1.213749e+10,False,False,US,35.181621,7022,784,116,34.039264,-118.260974


In [8]:
# save the sample dataframe to a parquet file
df.to_parquet("/home/Shared/datasets/humo_data/veraset/Visits/LosAngeles/subset_pois_veraset_processed.parquet", index=False)

### Generate eval sets

In [2]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from shapely.errors import WKTReadingError
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree
import os

/tmp/ipykernel_3251970/1319821888.py:6: DeprecationWarning: WKTReadingError is deprecated and will be removed in a future version. Use ShapelyError instead (functions previously raising {name} will now raise a ShapelyError instead).
  from shapely.errors import WKTReadingError


In [3]:
# load whole dataset
df = pd.read_parquet("/home/Shared/datasets/humo_data/veraset/Visits/LosAngeles/whole_veraset_processed.parquet")

df.shape

(6908365, 45)

In [16]:
print(df['is_closed'].value_counts())

is_closed
False    5233269
True     1675096
Name: count, dtype: int64


In [4]:
# print the number of unique POIs

print(f"Number of unique POIs: {df['place_id'].nunique()}")

Number of unique POIs: 39556


In [14]:
# print the number of unique POIs where is_closed is True

print(f"Number of unique POIs where is_closed is True: {df[df['closed_on']]['place_id'].nunique()}")

KeyError: 'closed_on'

In [13]:
df[df['closed_on']]['place_id'].unique()

KeyError: 'closed_on'

In [5]:
df.columns

Index(['user_str', 'arrival_time', 'departure_time', 'gps_coordinates',
       'geometry.latitude', 'geometry.longitude', 'safegraph.placekey',
       'safegraph.place_id', 'safegraph.parent_placekey',
       'safegraph.parent_place_id', 'safegraph.brand_ids',
       'safegraph.location_name', 'safegraph.brands', 'safegraph.top_category',
       'safegraph.sub_category', 'safegraph.naics_code', 'safegraph.latitude',
       'safegraph.longitude', 'safegraph.street_address', 'safegraph.city',
       'safegraph.region', 'safegraph.postal_code', 'safegraph.open_hours',
       'safegraph.category_tags', 'safegraph.opened_on', 'safegraph.closed_on',
       'safegraph.tracking_opened_since', 'safegraph.tracking_closed_since',
       'safegraph.polygon_wkt', 'safegraph.polygon_class',
       'safegraph.building_height', 'safegraph.enclosed',
       'safegraph.phone_number', 'safegraph.is_synthetic',
       'safegraph.includes_parking_lot', 'safegraph.iso_country_code',
       'safegraph.dist_t

In [6]:
print(df['safegraph.closed_on'].unique())

[None '2020-01' '2020-08' '2020-12' '2020-07' '2020-03' '2020-10'
 '2021-02' '2020-02' '2020-11' '2019-10' '2021-01' '2019-09' '2019-12'
 '2020-09' '2020-06' '2020-04' '2019-07' '2020-05' '2019-11']


In [3]:
import json
import numpy as np

def parse_open_hours_to_168(open_hours_str):
    """
    Converts SafeGraph-style open_hours string into a 168-dimensional binary vector.
    
    Args:
        open_hours_str (str): JSON-style string of weekly open hours.
        
    Returns:
        np.ndarray: A binary vector of shape (168,) where 1 indicates the POI is open.
    """
    # Initialize all hours to closed
    vector = np.zeros(168, dtype=np.uint8)

    if not isinstance(open_hours_str, str) or open_hours_str.strip() == '':
        return vector

    try:
        schedule = json.loads(open_hours_str)
    except json.JSONDecodeError:
        return vector

    day_to_index = {"Mon": 0, "Tue": 1, "Wed": 2,
                    "Thu": 3, "Fri": 4, "Sat": 5, "Sun": 6}

    for day, intervals in schedule.items():
        if day not in day_to_index:
            continue
        day_idx = day_to_index[day]

        for interval in intervals:
            if len(interval) != 2:
                continue
            try:
                start_hour = int(interval[0].split(':')[0])
                end_hour = int(interval[1].split(':')[0])
            except ValueError:
                continue

            base_idx = day_idx * 24

            if end_hour <= start_hour:
                # e.g., 22:00 to 2:00 spans two days
                vector[base_idx + start_hour:base_idx + 24] = 1
                next_day_idx = ((day_idx + 1) % 7) * 24
                vector[next_day_idx:next_day_idx + end_hour] = 1
            else:
                vector[base_idx + start_hour:base_idx + end_hour] = 1

    return vector

# Example usage
example_input = '{ "Mon": [["6:00", "22:00"]], "Tue": [], "Wed": [], "Thu": [], "Fri": [], "Sat": [], "Sun": [] }'
parsed_vector = parse_open_hours_to_168(example_input)

In [4]:
parsed_vector

array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], dtype=uint8)